# Tugas Deep Learning — CNN vs BiGRU pada Fashion-MNIST

**Nama:** Ryan Firmansyah  
**NIM:** H071241082  
**Program Studi:** Sistem Informasi, Departemen Matematika, FMIPA, Universitas Hasanuddin

## Tujuan
Mengimplementasikan dua arsitektur deep learning pada dataset yang sama, lalu membandingkannya:

1. **CNN (Convolutional Neural Network)**, membaca gambar sebagai grid 2D.
2. **RNN berbasis GRU (Bidirectional GRU)**, membaca gambar sebagai *sequence* 28 baris piksel (28 time step × 28 fitur).

**Dataset:** [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) (Zalando Research): 70.000 citra grayscale 28×28, 10 kelas pakaian (60.000 train, 10.000 test).

In [1]:
import os, gzip, json, time, urllib.request
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
keras.utils.set_random_seed(SEED)
EPOCHS = 12
BATCH = 128
os.makedirs("results", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

I0000 00:00:1794595602.038445     355 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794595602.098159     355 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1794595603.524444     355 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow 2.21.0 | Keras 3.15.1


## 1. Load Dataset
File dataset diunduh langsung dari repositori GitHub resmi Fashion-MNIST (folder `data/` tidak di-commit).

In [2]:
BASE = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
FILES = ["train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz",
         "t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"]
for f in FILES:
    p = os.path.join("data", f)
    if not os.path.exists(p):
        urllib.request.urlretrieve(BASE + f, p)

def read_images(p):
    with gzip.open(p, "rb") as fh:
        return np.frombuffer(fh.read(), np.uint8, offset=16).reshape(-1, 28, 28)

def read_labels(p):
    with gzip.open(p, "rb") as fh:
        return np.frombuffer(fh.read(), np.uint8, offset=8)

X_train_full = read_images("data/train-images-idx3-ubyte.gz")
y_train_full = read_labels("data/train-labels-idx1-ubyte.gz")
X_test = read_images("data/t10k-images-idx3-ubyte.gz")
y_test = read_labels("data/t10k-labels-idx1-ubyte.gz")

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
print("Train:", X_train_full.shape, y_train_full.shape)
print("Test :", X_test.shape, y_test.shape)

Train: (60000, 28, 28) (60000,)
Test : (10000, 28, 28) (10000,)


## 2. Eksplorasi Data

In [3]:
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for c in range(10):
    idx = np.where(y_train_full == c)[0][:2]
    for r, i in enumerate(idx):
        axes[r, c].imshow(X_train_full[i], cmap="gray")
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(CLASS_NAMES[c], fontsize=9)
plt.suptitle("Contoh citra per kelas", y=1.02)
plt.tight_layout()
plt.savefig("results/sample_images.png", dpi=130, bbox_inches="tight")
plt.show()

print("Distribusi kelas (train):", dict(zip(CLASS_NAMES, np.bincount(y_train_full))))
print("Distribusi kelas (test) :", dict(zip(CLASS_NAMES, np.bincount(y_test))))
print("Rentang piksel:", X_train_full.min(), "-", X_train_full.max())

Distribusi kelas (train): {'T-shirt/top': np.int64(6000), 'Trouser': np.int64(6000), 'Pullover': np.int64(6000), 'Dress': np.int64(6000), 'Coat': np.int64(6000), 'Sandal': np.int64(6000), 'Shirt': np.int64(6000), 'Sneaker': np.int64(6000), 'Bag': np.int64(6000), 'Ankle boot': np.int64(6000)}
Distribusi kelas (test) : {'T-shirt/top': np.int64(1000), 'Trouser': np.int64(1000), 'Pullover': np.int64(1000), 'Dress': np.int64(1000), 'Coat': np.int64(1000), 'Sandal': np.int64(1000), 'Shirt': np.int64(1000), 'Sneaker': np.int64(1000), 'Bag': np.int64(1000), 'Ankle boot': np.int64(1000)}
Rentang piksel: 0 - 255


## 3. Preprocessing
- Normalisasi piksel ke rentang [0, 1].
- Split validasi 10% dari data train secara *stratified* (54.000 train / 6.000 validasi).
- CNN menerima input `(28, 28, 1)`; BiGRU menerima input `(28, 28)` = 28 time step (baris) × 28 fitur (kolom).

In [4]:
Xf = X_train_full.astype("float32") / 255.0
Xt = X_test.astype("float32") / 255.0
X_tr, X_val, y_tr, y_val = train_test_split(
    Xf, y_train_full, test_size=0.1, stratify=y_train_full, random_state=SEED)

# CNN: tambah channel dimension
X_tr_c, X_val_c, X_te_c = X_tr[..., None], X_val[..., None], Xt[..., None]
print("CNN input :", X_tr_c.shape, X_val_c.shape, X_te_c.shape)
print("GRU input :", X_tr.shape, X_val.shape, Xt.shape)

CNN input : (54000, 28, 28, 1) (6000, 28, 28, 1) (10000, 28, 28, 1)
GRU input : (54000, 28, 28) (6000, 28, 28) (10000, 28, 28)


## 4. Model 1 — CNN
Dua blok konvolusi (Conv→BatchNorm→ReLU ×2, MaxPool, Dropout) dengan 32 dan 64 filter, kemudian lapisan Dense 128 unit dan softmax 10 kelas.

In [5]:
def build_cnn():
    m = keras.Sequential([
        layers.Input((28, 28, 1)),
        layers.Conv2D(32, 3, padding="same", use_bias=False), layers.BatchNormalization(), layers.ReLU(),
        layers.Conv2D(32, 3, padding="same", use_bias=False), layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPooling2D(), layers.Dropout(0.25),
        layers.Conv2D(64, 3, padding="same", use_bias=False), layers.BatchNormalization(), layers.ReLU(),
        layers.Conv2D(64, 3, padding="same", use_bias=False), layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPooling2D(), layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(128, activation="relu"), layers.Dropout(0.4),
        layers.Dense(10, activation="softmax"),
    ], name="CNN")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

cnn = build_cnn()
cnn.summary()

Model: "CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 28, 28, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 32)     │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 28, 28, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 14, 14, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 468,394 (1.79 MB)

 Trainable params: 468,010 (1.79 MB)

 Non-trainable params: 384 (1.50 KB)

In [6]:
def callbacks():
    return [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
    ]

t0 = time.time()
hist_cnn = cnn.fit(X_tr_c, y_tr, validation_data=(X_val_c, y_val),
                   epochs=EPOCHS, batch_size=BATCH, callbacks=callbacks(), verbose=2)
time_cnn = time.time() - t0
print(f"Waktu training CNN: {time_cnn:.0f} detik")

Epoch 1/12


422/422 - 66s - 157ms/step - accuracy: 0.7629 - loss: 0.6633 - val_accuracy: 0.6960 - val_loss: 0.8965 - learning_rate: 0.0010


Epoch 2/12


422/422 - 63s - 150ms/step - accuracy: 0.8489 - loss: 0.4154 - val_accuracy: 0.8952 - val_loss: 0.2822 - learning_rate: 0.0010


Epoch 3/12


422/422 - 61s - 145ms/step - accuracy: 0.8705 - loss: 0.3590 - val_accuracy: 0.9093 - val_loss: 0.2558 - learning_rate: 0.0010


... (15 outputs left out)

422/422 - 66s - 155ms/step - accuracy: 0.9209 - loss: 0.2170 - val_accuracy: 0.9358 - val_loss: 0.1730 - learning_rate: 5.0000e-04


Epoch 12/12


422/422 - 84s - 199ms/step - accuracy: 0.9242 - loss: 0.2087 - val_accuracy: 0.9283 - val_loss: 0.1990 - learning_rate: 5.0000e-04


Waktu training CNN: 912 detik


## 5. Model 2 — Bidirectional GRU (RNN)
Setiap gambar dibaca baris demi baris (28 time step, 28 fitur per step). Dua lapis GRU: BiGRU (96 unit, `return_sequences=True`) lalu GRU (64 unit), diikuti Dense softmax. Arah dua (*bidirectional*) membuat model melihat konteks atas→bawah dan bawah→atas.

In [7]:
def build_gru():
    m = keras.Sequential([
        layers.Input((28, 28)),
        layers.Bidirectional(layers.GRU(96, return_sequences=True)),
        layers.Dropout(0.25),
        layers.GRU(64),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="BiGRU")
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

gru = build_gru()
gru.summary()

Model: "BiGRU"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 28, 192)        │        72,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 28, 192)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        49,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 126,922 (495.79 KB)

 Trainable params: 126,922 (495.79 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
t0 = time.time()
hist_gru = gru.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                   epochs=EPOCHS, batch_size=BATCH, callbacks=callbacks(), verbose=2)
time_gru = time.time() - t0
print(f"Waktu training BiGRU: {time_gru:.0f} detik")

Epoch 1/12


422/422 - 32s - 76ms/step - accuracy: 0.7045 - loss: 0.7901 - val_accuracy: 0.8187 - val_loss: 0.5052 - learning_rate: 0.0010


Epoch 2/12


422/422 - 27s - 64ms/step - accuracy: 0.8249 - loss: 0.4845 - val_accuracy: 0.8572 - val_loss: 0.4018 - learning_rate: 0.0010


Epoch 3/12


422/422 - 41s - 96ms/step - accuracy: 0.8475 - loss: 0.4215 - val_accuracy: 0.8657 - val_loss: 0.3749 - learning_rate: 0.0010


... (15 outputs left out)

422/422 - 40s - 96ms/step - accuracy: 0.8962 - loss: 0.2815 - val_accuracy: 0.8973 - val_loss: 0.2774 - learning_rate: 0.0010


Epoch 12/12


422/422 - 39s - 92ms/step - accuracy: 0.8994 - loss: 0.2714 - val_accuracy: 0.9022 - val_loss: 0.2701 - learning_rate: 0.0010


Waktu training BiGRU: 469 detik


## 6. Kurva Training

In [9]:
def plot_hist(h, name, ax):
    ax[0].plot(h.history["accuracy"], label="train"); ax[0].plot(h.history["val_accuracy"], label="val")
    ax[0].set_title(f"{name} — Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(h.history["loss"], label="train"); ax[1].plot(h.history["val_loss"], label="val")
    ax[1].set_title(f"{name} — Loss"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
plot_hist(hist_cnn, "CNN", ax[0]); plot_hist(hist_gru, "BiGRU", ax[1])
plt.tight_layout(); plt.savefig("results/training_curves.png", dpi=130); plt.show()

## 7. Evaluasi pada Data Test

In [10]:
results = {}
preds = {}
for name, model, Xte, ttrain, hist in [("CNN", cnn, X_te_c, time_cnn, hist_cnn),
                                       ("BiGRU", gru, Xt, time_gru, hist_gru)]:
    t0 = time.time()
    prob = model.predict(Xte, batch_size=256, verbose=0)
    t_inf = time.time() - t0
    y_pred = prob.argmax(1)
    preds[name] = y_pred
    rep = classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True, digits=4)
    results[name] = {
        "test_accuracy": float((y_pred == y_test).mean()),
        "macro_f1": float(rep["macro avg"]["f1-score"]),
        "best_val_accuracy": float(max(hist.history["val_accuracy"])),
        "epochs_run": len(hist.history["loss"]),
        "params": int(model.count_params()),
        "train_time_sec": round(ttrain, 1),
        "inference_time_sec": round(t_inf, 2),
        "per_class_f1": {c: round(rep[c]["f1-score"], 4) for c in CLASS_NAMES},
    }
    print(f"===== {name} =====")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

with open("results/metrics.json", "w") as f:
    json.dump(results, f, indent=2)

===== CNN =====
              precision    recall  f1-score   support

 T-shirt/top     0.8573    0.9130    0.8843      1000
     Trouser     0.9980    0.9760    0.9869      1000
    Pullover     0.8813    0.9060    0.8935      1000
       Dress     0.8903    0.9410    0.9149      1000
        Coat     0.8899    0.8970    0.8934      1000
      Sandal     0.9909    0.9820    0.9864      1000
       Shirt     0.8308    0.7220    0.7726      1000
     Sneaker     0.9505    0.9790    0.9645      1000
         Bag     0.9930    0.9870    0.9900      1000
  Ankle boot     0.9786    0.9590    0.9687      1000

    accuracy                         0.9262     10000
   macro avg     0.9260    0.9262    0.9255     10000
weighted avg     0.9260    0.9262    0.9255     10000



===== BiGRU =====
              precision    recall  f1-score   support

 T-shirt/top     0.8439    0.8220    0.8328      1000
     Trouser     0.9828    0.9710    0.9769      1000
    Pullover     0.8320    0.8020    0.8167      1000
       Dress     0.8210    0.9310    0.8725      1000
        Coat     0.8236    0.8170    0.8203      1000
      Sandal     0.9740    0.9730    0.9735      1000
       Shirt     0.7298    0.6860    0.7072      1000
     Sneaker     0.9531    0.9560    0.9546      1000
         Bag     0.9731    0.9750    0.9740      1000
  Ankle boot     0.9631    0.9670    0.9651      1000

    accuracy                         0.8900     10000
   macro avg     0.8896    0.8900    0.8894     10000
weighted avg     0.8896    0.8900    0.8894     10000



In [11]:
import pandas as pd
cmp = pd.DataFrame({
    n: {"Test accuracy": f"{r['test_accuracy']*100:.2f}%", "Macro F1": f"{r['macro_f1']:.4f}",
        "Best val acc": f"{r['best_val_accuracy']*100:.2f}%", "Parameter": f"{r['params']:,}",
        "Epoch dijalankan": r["epochs_run"], "Waktu training (s)": r["train_time_sec"]}
    for n, r in results.items()}).T
cmp

,Test accuracy,Macro F1,Best val acc,Parameter,Epoch dijalankan,Waktu training (s)
CNN,92.62%,0.9255,93.58%,"468,394",12,912.4
BiGRU,89.00%,0.8894,90.22%,"126,922",12,468.8


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
for ax, name in zip(axes, ["CNN", "BiGRU"]):
    cm = confusion_matrix(y_test, preds[name])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"Confusion Matrix — {name} (acc {results[name]['test_accuracy']*100:.2f}%)")
    ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual")
    ax.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.savefig("results/confusion_matrices.png", dpi=130); plt.show()

## 8. Analisis Kesalahan

In [13]:
# Pasangan kelas yang paling sering tertukar
for name in ["CNN", "BiGRU"]:
    cm = confusion_matrix(y_test, preds[name]).copy()
    np.fill_diagonal(cm, 0)
    top = np.dstack(np.unravel_index(np.argsort(cm.ravel())[::-1][:3], cm.shape))[0]
    print(f"{name}: kesalahan terbanyak")
    for a, b in top:
        print(f"   {CLASS_NAMES[a]:12s} -> diprediksi {CLASS_NAMES[b]:12s}: {cm[a, b]} gambar")

# Contoh gambar salah prediksi (CNN)
wrong = np.where(preds["CNN"] != y_test)[0][:10]
fig, axes = plt.subplots(2, 5, figsize=(12, 5.5))
for ax, i in zip(axes.ravel(), wrong):
    ax.imshow(X_test[i], cmap="gray"); ax.axis("off")
    ax.set_title(f"Aktual: {CLASS_NAMES[y_test[i]]}\nCNN: {CLASS_NAMES[preds['CNN'][i]]}", fontsize=9)
plt.tight_layout(); plt.savefig("results/cnn_misclassified.png", dpi=130); plt.show()

both = (preds["CNN"] != y_test) & (preds["BiGRU"] != y_test)
print(f"Salah di kedua model: {both.sum()} | hanya CNN salah: {((preds['CNN']!=y_test)&(preds['BiGRU']==y_test)).sum()} | hanya BiGRU salah: {((preds['CNN']==y_test)&(preds['BiGRU']!=y_test)).sum()}")

CNN: kesalahan terbanyak
   Shirt        -> diprediksi T-shirt/top : 122 gambar
   Shirt        -> diprediksi Coat        : 60 gambar
   Shirt        -> diprediksi Pullover    : 58 gambar
BiGRU: kesalahan terbanyak
   Shirt        -> diprediksi T-shirt/top : 121 gambar
   T-shirt/top  -> diprediksi Shirt       : 98 gambar
   Coat         -> diprediksi Pullover    : 85 gambar


Salah di kedua model: 540 | hanya CNN salah: 198 | hanya BiGRU salah: 560


In [14]:
cnn.save("results/cnn_fashion_mnist.keras")
gru.save("results/bigru_fashion_mnist.keras")
print({k: os.path.getsize(f"results/{k}") // 1024 for k in os.listdir("results") if k.endswith(".keras")}, "KB")

{'bigru_fashion_mnist.keras': 1538, 'cnn_fashion_mnist.keras': 5567} KB


## 9. Kesimpulan

| Model | Test accuracy | Macro F1 | Parameter | Waktu training (CPU 2 core) |
|---|---|---|---|---|
| CNN | **92,62%** | 0,9255 | 468.394 | 912 s |
| BiGRU | 89,00% | 0,8894 | 126.922 | 469 s |

1. **CNN lebih akurat (+3,6 poin)** daripada BiGRU pada Fashion-MNIST. Konvolusi 2D memang dirancang menangkap pola spasial lokal (tepi, tekstur, bentuk) pada gambar, sedangkan GRU hanya melihat gambar sebagai urutan baris sehingga struktur 2D harus dipelajari secara tidak langsung.
2. **BiGRU jauh lebih ringan** (±127 ribu parameter, sekitar 3,7× lebih sedikit, dan training sekitar 2× lebih cepat di CPU), tetapi akurasinya lebih rendah. Kurva training menunjukkan BiGRU masih membaik pada epoch ke-12, jadi dengan epoch lebih banyak selisihnya kemungkinan mengecil; hal ini tidak diuji di sini.
3. **Kelas tersulit sama untuk kedua model: Shirt** (F1 CNN 0,77; BiGRU 0,71), yang paling sering tertukar dengan T-shirt/top, Coat, dan Pullover. Bentuk keempat kelas ini mirip pada resolusi 28×28.
4. Tidak ada tanda overfitting berat: akurasi validasi sebanding atau sedikit di atas train karena Dropout/BatchNorm aktif saat training. Satu kali run dengan seed 42, jadi selisih kecil antar-run tidak diukur.
5. Model Transfer Learning tidak dikerjakan karena bobot pretrained tidak dapat diunduh di lingkungan yang dipakai; tugas hanya mensyaratkan minimal 2 arsitektur, sehingga CNN + RNN (GRU) memenuhi ketentuan.
